# 도구 검색: 대안적 접근법

**먼저 "Tool Search with Embeddings" 쿡북을 읽어 보시길 권합니다.**

이 쿡북의 목표는 Claude에서 도구 검색(정확히는 "도구 발견")을 구현하는 몇 가지 대안적 접근법을 소개하는 것입니다. 여기서는 유용한 두 가지 기법을 다룹니다.

1. 도구는 "검색" 없이도 발견할 수 있습니다. 이 쿡북에서는 모든 도구 이름을 Claude의 시스템 프롬프트에 넣고, describe_tool_tool을 제공해 도구 전체를 Claude의 컨텍스트로 불러오게 합니다.
2. 아직 Claude의 컨텍스트에 적재되지 않은 도구라면 요청의 `tools` 목록에 넣지 않아도 됩니다. 애플리케이션에서 관리할 복잡도는 조금 늘어나지만, Claude가 잠재적으로 수천 개의 도구에 접근할 수 있는 상황에서도 요청을 작게 유지할 수 있습니다.

사용자는 Claude의 컨텍스트(그리고 Messages 요청)를 최대한 집중된 상태로 유지하도록 도구 검색을 설계할 자유도가 큽니다.

## 사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**
- Python 기초 — 함수, 딕셔너리, 기본 자료구조에 익숙할 것
- Claude 도구 사용에 대한 기본 이해 — 먼저 [도구 사용 가이드](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)를 읽어 보시길 권합니다

**필요한 도구**
- Python 3.11 이상
- Anthropic API 키 ([여기서 발급](https://docs.anthropic.com/claude/reference/getting-started-with-the-api))

In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import json

import anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic()

print("✓ Client initialized")

## 도구 라이브러리 정의하기

간단한 도구 5개를 정의하겠습니다. 실제 프로덕션에서는 데이터베이스나 설정 파일에 저장된 수백, 수천 개의 도구가 될 수 있습니다.

In [ ]:
TOOL_LIBRARY = {
    "get_weather": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    },
    "get_stock_price": {
        "name": "get_stock_price",
        "description": "Get current stock price for a ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker (e.g., AAPL)"},
            },
            "required": ["ticker"],
        },
    },
    "convert_currency": {
        "name": "convert_currency",
        "description": "Convert amount between currencies",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    "calculate_tip": {
        "name": "calculate_tip",
        "description": "Calculate tip amount for a bill",
        "input_schema": {
            "type": "object",
            "properties": {
                "bill_amount": {"type": "number"},
                "tip_percent": {"type": "number", "default": 20},
            },
            "required": ["bill_amount"],
        },
    },
    "send_email": {
        "name": "send_email",
        "description": "Send an email to a recipient",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
}

print(f"✓ Defined {len(TOOL_LIBRARY)} tools: {list(TOOL_LIBRARY.keys())}")

## `describe_tool` 도구

시맨틱 검색 대신, Claude에 간단한 `describe_tool` 도구를 제공합니다. Claude가 도구 이름을 인자로 이 도구를 호출하면 해당 도구가 컨텍스트로 적재됩니다.

시스템 프롬프트에 사용 가능한 도구 이름이 모두 나열되어 있으므로, 임베딩이나 검색 없이도 Claude는 어떤 도구가 있는지 알 수 있습니다.

In [ ]:
DESCRIBE_TOOL = {
    "name": "describe_tool",
    "description": "Load a tool's full definition into context. Call this before using any tool for the first time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tool_name": {
                "type": "string",
                "description": "Name of the tool to load",
            },
        },
        "required": ["tool_name"],
    },
}

# Build system prompt with tool catalog
tool_names = list(TOOL_LIBRARY.keys())
SYSTEM_PROMPT = f"""You are a helpful assistant with access to various tools.

Available tools: {", ".join(tool_names)}

Before using any tool, you must first call describe_tool with the tool name to load it."""

print("System prompt:")
print(SYSTEM_PROMPT)

## 목(mock) 도구 실행

시연을 위한 간단한 목 응답입니다:

In [ ]:
def execute_tool(name: str, inputs: dict) -> str:
    """Mock tool execution."""
    if name == "get_weather":
        return json.dumps({"city": inputs["city"], "temp": "72°F", "conditions": "Sunny"})
    elif name == "get_stock_price":
        return json.dumps({"ticker": inputs["ticker"], "price": 185.50, "change": "+1.2%"})
    elif name == "convert_currency":
        rate = 0.92 if inputs["to_currency"] == "EUR" else 1.0
        converted = inputs["amount"] * rate
        return json.dumps({"converted": round(converted, 2), "to": inputs["to_currency"]})
    elif name == "calculate_tip":
        tip = inputs["bill_amount"] * (inputs.get("tip_percent", 20) / 100)
        return json.dumps({"tip": round(tip, 2), "total": round(inputs["bill_amount"] + tip, 2)})
    elif name == "send_email":
        return json.dumps({"status": "sent", "to": inputs["to"]})
    return json.dumps({"error": f"Unknown tool: {name}"})


print("✓ Mock execution ready")

## 동적 도구 적재를 사용하는 대화 루프

여기서의 핵심 패턴은 다음과 같습니다.

1. **tools 목록에는 `describe_tool`만 넣고 시작합니다**
2. **Claude가 `describe_tool`을 호출하면**, `tool_reference`를 반환하고 동시에 해당 도구를 `defer_loading=True`로 `active_tools`에 추가합니다
3. **`defer_loading=True`가 결정적입니다** — 도구 정의가 캐싱된 프롬프트 접두부에 들어가지 않게 하여, 도구가 새로 발견될 때 캐시가 무효화되는 것을 막아 줍니다

대화 중 Claude가 `tool_reference`를 볼 때마다, 대화의 바로 그 지점에서 전체 도구 정의가 Claude의 컨텍스트로 적재됩니다.

In [ ]:
def run_conversation(user_message: str, max_turns: int = 10):
    """Run a conversation with dynamic tool loading."""
    print(f"\n{'=' * 60}")
    print(f"USER: {user_message}")
    print(f"{'=' * 60}\n")

    messages = [{"role": "user", "content": user_message}]

    # Start with ONLY describe_tool - no other tools in the request
    active_tools = [DESCRIBE_TOOL]
    loaded_tools = set()  # Track which tools we've added

    for turn in range(max_turns):
        print(f"--- Turn {turn + 1} (tools in request: {len(active_tools)}) ---")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=active_tools,
            messages=messages,
            extra_headers={"anthropic-beta": "advanced-tool-use-2025-11-20"},
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    print(f"\nASSISTANT: {block.text}")
            break

        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == "text" and block.text:
                print(f"ASSISTANT: {block.text}")

            elif block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input

                if tool_name == "describe_tool":
                    requested_tool = tool_input["tool_name"]
                    print(f"🔍 describe_tool({requested_tool})")

                    if requested_tool in TOOL_LIBRARY:
                        # Add tool to active_tools with defer_loading=True
                        # This is critical for prompt caching!
                        if requested_tool not in loaded_tools:
                            tool_def = {**TOOL_LIBRARY[requested_tool], "defer_loading": True}
                            active_tools.append(tool_def)
                            loaded_tools.add(requested_tool)
                            print(f"   ✓ Added {requested_tool} to tools (defer_loading=True)")

                        # Return tool_reference so Claude can use it
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": [
                                    {"type": "tool_reference", "tool_name": requested_tool}
                                ],
                            }
                        )
                    else:
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": f"Tool '{requested_tool}' not found.",
                            }
                        )
                else:
                    # Execute discovered tool
                    print(f"🔧 {tool_name}({json.dumps(tool_input)})")
                    result = execute_tool(tool_name, tool_input)
                    print(f"   → {result}")
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )

        if tool_results:
            messages.append({"role": "user", "content": tool_results})

    print(f"\n{'=' * 60}\n")


print("✓ Conversation loop ready")

## 예제: 날씨 질의

Claude가 다음 순서로 동작하는 것을 확인해 보세요.
1. 시스템 프롬프트의 도구 목록에서 `get_weather`를 확인합니다
2. `describe_tool("get_weather")`를 호출해 도구를 적재합니다
3. `tool_reference`를 받아 이제 그 도구를 사용할 수 있게 됩니다

In [ ]:
run_conversation("What's the weather in Tokyo?")

## 예제: 여러 도구가 필요한 질의

Claude는 필요에 따라 여러 도구를 적재할 수 있습니다:

In [ ]:
run_conversation("Convert $100 to EUR, then calculate a 20% tip on a $85 dinner bill.")

## `defer_loading=True`가 중요한 이유

도구를 `tools` 목록에 추가하면, 도구 정의는 보통 Claude 컨텍스트 윈도의 맨 앞에 적재됩니다. 새 도구를 추가하면 컨텍스트 윈도의 맨 앞이 바뀌기 때문에 캐시의 대부분을 잃게 됩니다.

**`defer_loading=True`를 사용하면:**
- 도구 정의가 Claude 컨텍스트 윈도의 맨 앞에 포함되지 않습니다.
- 대신 Claude가 `tool_reference`를 보는 시점에 컨텍스트로 적재됩니다
- 즉 새 도구를 발견하더라도 시스템 프롬프트와 초기 도구들은 캐싱된 상태로 유지됩니다

**패턴:**
```python
# Initial request - only describe_tool, system prompt is cached
tools = [DESCRIBE_TOOL]

# After Claude calls describe_tool("get_weather")
# Add with defer_loading to preserve cache
tools.append({**TOOL_LIBRARY["get_weather"], "defer_loading": True})

# Return tool_reference so Claude knows it's available
tool_result = [{"type": "tool_reference", "tool_name": "get_weather"}]
```

수백, 수천 개의 도구를 다루면서 다음을 원하는 애플리케이션에는 이 방식이 필수적입니다.
- 초기 요청 크기를 작게 유지하기
- 도구를 발견해 나가는 동안에도 프롬프트 캐싱 유지하기
- Claude가 실제로 필요로 하는 도구만 적재하기

## 마무리

이 쿡북의 핵심은 이것입니다. **도구는 Claude가 필요로 하기 전까지 `tools` 목록에 있을 필요가 없습니다.** `defer_loading=True` 및 `tool_reference`와 결합하면, 요청을 작게 유지하고 프롬프트 캐싱을 보존하면서도 수천 개의 도구로 확장할 수 있습니다. 다만 이 경우 클라이언트 쪽에서 도구 발견 메커니즘을 제공해야 합니다.

여기서 보여 준 `describe_tool` 방식은 여러 방법 중 하나일 뿐입니다. 다른 패턴으로는 다음과 같은 것들이 있습니다.
- **`list_tools`** — 특정 범주나 키워드에 해당하는 도구 이름을 반환
- **계층적 발견** — 도구 범주를 탐색한 뒤 특정 도구를 적재
- **하이브리드** — 대규모 카탈로그를 위해 목록 조회와 시맨틱 검색을 결합

핵심 패턴은 언제나 동일합니다.
1. 도구가 발견되면 `tool_reference`를 반환합니다
2. 캐싱을 보존하기 위해 `defer_loading=True`로 도구를 추가합니다
3. 그러면 Claude가 곧바로 그 도구를 사용할 수 있습니다

임베딩 기반 접근법은 [Tool Search with Embeddings](./tool_search_with_embeddings.ipynb)를 참고하세요.